# Демо 1. Класичне донавчання: зміна голови в моделі BERT

У цьому ноутбуці донавчаємо модель **BERT** для задачі **визначення парафраз**: модель отримує два речення й визначає, чи передають вони той самий зміст.

План:
1. Завантажити датасет MRPC і токенізувати пари речень.
2. Завантажити BERT із **новою головою**-класифікатором.
3. **Повне донавчання** з `Trainer`: тренуються всі ваги (кістяк і голова).
4. Перевірити модель на власних прикладах.

Приклад за мотивами розділу 3 курсу [Hugging Face LLM Course](https://huggingface.co/learn/llm-course/chapter3/1).
Для запуску в Colab оберіть середовище з GPU: *Runtime → Change runtime type → T4 GPU*.

In [1]:
import numpy as np
import torch
from datasets import load_dataset
from sklearn.metrics import accuracy_score, f1_score
from transformers import (
    AutoModelForSequenceClassification,
    AutoTokenizer,
    DataCollatorWithPadding,
    Trainer,
    TrainingArguments,
    set_seed,
)

In [3]:
# Фіксуємо seed, щоб результати повторювалися між запусками
set_seed(42)

device = "cuda" if torch.cuda.is_available() else "cpu"
print(f"Пристрій: {device}")
if device == "cuda":
    print(f"GPU: {torch.cuda.get_device_name()}")
else:
    print("Включіть GPU, бо набридне чекати ;)")

Пристрій: cuda
GPU: Tesla T4


## 1. Дані: MRPC

**MRPC** (Microsoft Research Paraphrase Corpus) — пари речень з новинних статей.
Мітка `1` означає, що речення є парафразами, мітка `0` — що ні.

In [6]:
raw_datasets = load_dataset("nyu-mll/glue", "mrpc")
raw_datasets

README.md:   0%|          | 0.00/35.3k [00:00<?, ?B/s]

mrpc/train-00000-of-00001.parquet: reconstructing file:   0%|          |  0.00B /  649kB            

mrpc/train-00000-of-00001.parquet: downloading bytes:           |  0.00B            

mrpc/validation-00000-of-00001.parquet: reconstructing file:   0%|          |  0.00B / 75.7kB            

mrpc/validation-00000-of-00001.parquet: downloading bytes:           |  0.00B            

mrpc/test-00000-of-00001.parquet: reconstructing file:   0%|          |  0.00B /  308kB            

mrpc/test-00000-of-00001.parquet: downloading bytes:           |  0.00B            

Generating train split:   0%|          | 0/3668 [00:00<?, ? examples/s]

Generating validation split:   0%|          | 0/408 [00:00<?, ? examples/s]

Generating test split:   0%|          | 0/1725 [00:00<?, ? examples/s]

DatasetDict({
    train: Dataset({
        features: ['sentence1', 'sentence2', 'label', 'idx'],
        num_rows: 3668
    })
    validation: Dataset({
        features: ['sentence1', 'sentence2', 'label', 'idx'],
        num_rows: 408
    })
    test: Dataset({
        features: ['sentence1', 'sentence2', 'label', 'idx'],
        num_rows: 1725
    })
})

In [7]:
# Подивимося на один приклад
print("Приклад нееквівалентних речень:")
example = raw_datasets["train"][3]
print("Речення 1:", example["sentence1"])
print("Речення 2:", example["sentence2"])
print("Мітка:   ", example["label"], "→", raw_datasets["train"].features["label"].names[example["label"]])

print("")
print("Приклад еквівалентних речень:")
example = raw_datasets["train"][5]
print("Речення 1:", example["sentence1"])
print("Речення 2:", example["sentence2"])
print("Мітка:   ", example["label"], "→", raw_datasets["train"].features["label"].names[example["label"]])

Приклад нееквівалентних речень:
Речення 1: Around 0335 GMT , Tab shares were up 19 cents , or 4.4 % , at A $ 4.56 , having earlier set a record high of A $ 4.57 .
Речення 2: Tab shares jumped 20 cents , or 4.6 % , to set a record closing high at A $ 4.57 .
Мітка:    0 → not_equivalent

Приклад еквівалентних речень:
Речення 1: Revenue in the first quarter of the year dropped 15 percent from the same period a year earlier .
Речення 2: With the scandal hanging over Stewart 's company , revenue the first quarter of the year dropped 15 percent from the same period a year earlier .
Мітка:    1 → equivalent


### Знайомимось з BERT BASE UNCASED

Для цього прочитайте інформацію на [картці моделі](https://huggingface.co/google-bert/bert-base-uncased)

In [8]:
from transformers import pipeline

unmasker = pipeline('fill-mask', model='bert-base-uncased')
unmasker("The man worked as a [MASK].")

model.safetensors: reconstructing file:   0%|          |  0.00B /  440MB            

model.safetensors: downloading bytes:           |  0.00B            

Loading weights:   0%|          | 0/202 [00:00<?, ?it/s]

[transformers] BertForMaskedLM LOAD REPORT from: bert-base-uncased
Key                         | Status     |  | 
----------------------------+------------+--+-
bert.pooler.dense.weight    | UNEXPECTED |  | 
bert.pooler.dense.bias      | UNEXPECTED |  | 
cls.seq_relationship.weight | UNEXPECTED |  | 
cls.seq_relationship.bias   | UNEXPECTED |  | 

Notes:
- UNEXPECTED:	can be ignored when loading from different task/architecture; not ok if you expect identical arch.


[{'score': 0.09747552871704102,
  'token': 10533,
  'token_str': 'carpenter',
  'sequence': 'the man worked as a carpenter.'},
 {'score': 0.052383244037628174,
  'token': 15610,
  'token_str': 'waiter',
  'sequence': 'the man worked as a waiter.'},
 {'score': 0.049627017229795456,
  'token': 13362,
  'token_str': 'barber',
  'sequence': 'the man worked as a barber.'},
 {'score': 0.037886038422584534,
  'token': 15893,
  'token_str': 'mechanic',
  'sequence': 'the man worked as a mechanic.'},
 {'score': 0.037680864334106445,
  'token': 18968,
  'token_str': 'salesman',
  'sequence': 'the man worked as a salesman.'}]

In [9]:
print("Не по темі, але погляньте, яка \"несправедливість\"...")
unmasker("The woman worked as a [MASK].")

Не по темі, але погляньте, яка "несправедливість"...


[{'score': 0.21981368958950043,
  'token': 6821,
  'token_str': 'nurse',
  'sequence': 'the woman worked as a nurse.'},
 {'score': 0.15973998606204987,
  'token': 13877,
  'token_str': 'waitress',
  'sequence': 'the woman worked as a waitress.'},
 {'score': 0.11547233909368515,
  'token': 10850,
  'token_str': 'maid',
  'sequence': 'the woman worked as a maid.'},
 {'score': 0.03796861320734024,
  'token': 19215,
  'token_str': 'prostitute',
  'sequence': 'the woman worked as a prostitute.'},
 {'score': 0.030423680320382118,
  'token': 5660,
  'token_str': 'cook',
  'sequence': 'the woman worked as a cook.'}]

### Токенізація пари речень

BERT приймає обидва речення як одну послідовність: `[CLS] речення 1 [SEP] речення 2 [SEP]`.
Поле `token_type_ids` позначає, до якого речення належить кожен токен (0 або 1).

Примітка: класифікаційний токен (`[CLS]`) використовується як вхідне представлення для задач класифікації в NLP та машинному навчанні. Він акумулює інформацію з усієї вхідної послідовності та передає її крізь шари моделі для подальшої обробки. Потім модель використовує це представлення для формування прогнозів або віднесення вхідних даних до заздалегідь визначених категорій.

In [10]:
checkpoint = "bert-base-uncased"

tokenizer = AutoTokenizer.from_pretrained(checkpoint)
encoded = tokenizer(example["sentence1"], example["sentence2"])

print("")
print(tokenizer.convert_ids_to_tokens(encoded["input_ids"])[:20])
print(encoded["token_type_ids"][:20])
print(tokenizer.convert_ids_to_tokens(encoded["input_ids"])[-20:])
print(encoded["token_type_ids"][-20:])


['[CLS]', 'revenue', 'in', 'the', 'first', 'quarter', 'of', 'the', 'year', 'dropped', '15', 'percent', 'from', 'the', 'same', 'period', 'a', 'year', 'earlier', '.']
[0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0]
[',', 'revenue', 'the', 'first', 'quarter', 'of', 'the', 'year', 'dropped', '15', 'percent', 'from', 'the', 'same', 'period', 'a', 'year', 'earlier', '.', '[SEP]']
[1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1]


In [11]:
def tokenize_function(batch):
    # truncation=True обрізає надто довгі пари до максимальної довжини моделі
    return tokenizer(batch["sentence1"], batch["sentence2"], truncation=True)


tokenized_datasets = raw_datasets.map(tokenize_function, batched=True)

# Доповнення (padding) робимо динамічно для кожного батча — до найдовшого прикладу в ньому
data_collator = DataCollatorWithPadding(tokenizer=tokenizer)
tokenized_datasets

Map:   0%|          | 0/3668 [00:00<?, ? examples/s]

Map:   0%|          | 0/408 [00:00<?, ? examples/s]

Map:   0%|          | 0/1725 [00:00<?, ? examples/s]

DatasetDict({
    train: Dataset({
        features: ['sentence1', 'sentence2', 'label', 'idx', 'input_ids', 'token_type_ids', 'attention_mask'],
        num_rows: 3668
    })
    validation: Dataset({
        features: ['sentence1', 'sentence2', 'label', 'idx', 'input_ids', 'token_type_ids', 'attention_mask'],
        num_rows: 408
    })
    test: Dataset({
        features: ['sentence1', 'sentence2', 'label', 'idx', 'input_ids', 'token_type_ids', 'attention_mask'],
        num_rows: 1725
    })
})

## 2. Модель: кістяк + нова голова

`AutoModelForSequenceClassification` бере кістяк (backbone) BERT і додає нову голову-класифікатор
з `num_labels` виходами. Зверніть увагу на звіт під час завантаження:
- ваги `cls.*` позначені як не використані (`UNEXPECTED`, у старих версіях — «not used»): це відкинута голова попереднього тренування;
- ваги `classifier.*` позначені як відсутні (`MISSING`, у старих версіях — «newly initialized»): це нова голова, ініціалізована випадково.

In [12]:
model = AutoModelForSequenceClassification.from_pretrained(checkpoint, num_labels=2)

print(model.classifier)

Loading weights:   0%|          | 0/199 [00:00<?, ?it/s]

[transformers] BertForSequenceClassification LOAD REPORT from: bert-base-uncased
Key                                        | Status     | 
-------------------------------------------+------------+-
cls.seq_relationship.weight                | UNEXPECTED | 
cls.predictions.transform.LayerNorm.weight | UNEXPECTED | 
cls.predictions.transform.dense.weight     | UNEXPECTED | 
cls.predictions.transform.dense.bias       | UNEXPECTED | 
cls.predictions.transform.LayerNorm.bias   | UNEXPECTED | 
cls.seq_relationship.bias                  | UNEXPECTED | 
cls.predictions.bias                       | UNEXPECTED | 
classifier.bias                            | MISSING    | 
classifier.weight                          | MISSING    | 

Notes:
- UNEXPECTED:	can be ignored when loading from different task/architecture; not ok if you expect identical arch.
- MISSING:	those params were newly initialized because missing from the checkpoint. Consider training on your downstream task.


Linear(in_features=768, out_features=2, bias=True)


In [13]:
def count_params(module, trainable_only=False):
    return sum(p.numel() for p in module.parameters() if p.requires_grad or not trainable_only)


print(f"Кістяк (model.bert):   {count_params(model.bert):>12,} параметрів")
print(f"Голова (model.classifier): {count_params(model.classifier):>8,} параметрів")

Кістяк (model.bert):    109,482,240 параметрів
Голова (model.classifier):    1,538 параметрів


## 3. Повне донавчання з `Trainer`

`Trainer` — клас бібліотеки Transformers, який містить увесь цикл тренування й оцінювання.
Метрики рахуємо самі у функції `compute_metrics`.

Оновлюються **всі** ваги: і кістяк, і нова голова.
Для повного донавчання використовують малий learning rate (`2e-5`),
щоб не зруйнувати знання, отримані під час попереднього тренування.

In [14]:
def compute_metrics(eval_pred):
    logits, labels = eval_pred
    predictions = np.argmax(logits, axis=-1)
    return {
        "accuracy": accuracy_score(labels, predictions),
        "f1": f1_score(labels, predictions),
    }


training_args = TrainingArguments(
    output_dir="bert-mrpc",
    num_train_epochs=3,             # зазвичай вистачає 1-3 епохи
    learning_rate=2e-5,
    per_device_train_batch_size=16,
    per_device_eval_batch_size=64,
    eval_strategy="epoch",          # оцінюємо на валідації після кожної епохи
    save_strategy="no",             # чекпоінти не зберігаємо
    logging_steps=50,
    fp16=torch.cuda.is_available(), # змішана точність пришвидшує тренування на GPU
    report_to="none",
    seed=42,
)

trainer = Trainer(
    model=model,
    args=training_args,
    train_dataset=tokenized_datasets["train"],
    eval_dataset=tokenized_datasets["validation"],
    data_collator=data_collator,
    compute_metrics=compute_metrics,
)

In [15]:
print(f"Параметрів, що тренуються: {count_params(model, trainable_only=True):,}")
trainer.train()
trainer.evaluate()

Параметрів, що тренуються: 109,483,778


Epoch,Training Loss,Validation Loss,Accuracy,F1
1,0.487028,0.381455,0.838235,0.886986
2,0.328676,0.357605,0.848039,0.893471
3,0.167169,0.416541,0.860294,0.901554


Training Loss,Validation Loss,Epoch,Accuracy,F1
0.167169,0.416541,3,0.860294,0.901554


{'eval_loss': 0.4165409803390503,
 'eval_accuracy': 0.8602941176470589,
 'eval_f1': 0.9015544041450777}

In [16]:
# Питання 9: Донавчання на 10 епохах (демонстрація перенавчання / overfitting)
set_seed(42)

# Завантажуємо чисту модель заново, щоб тренувати з нуля, а не поверх попередніх 3 епох
model_overfit = AutoModelForSequenceClassification.from_pretrained(checkpoint, num_labels=2)

training_args_10 = TrainingArguments(
    output_dir="bert-mrpc-10epochs",
    num_train_epochs=10,            # Збільшили кількість епох з 3 до 10
    learning_rate=2e-5,
    per_device_train_batch_size=16,
    per_device_eval_batch_size=64,
    eval_strategy="epoch",
    save_strategy="no",
    logging_steps=50,
    fp16=torch.cuda.is_available(),
    report_to="none",
    seed=42,
)

trainer_10 = Trainer(
    model=model_overfit,
    args=training_args_10,
    train_dataset=tokenized_datasets["train"],
    eval_dataset=tokenized_datasets["validation"],
    data_collator=data_collator,
    compute_metrics=compute_metrics,
)

trainer_10.train()
trainer_10.evaluate()

Loading weights:   0%|          | 0/199 [00:00<?, ?it/s]

[transformers] BertForSequenceClassification LOAD REPORT from: bert-base-uncased
Key                                        | Status     | 
-------------------------------------------+------------+-
cls.seq_relationship.weight                | UNEXPECTED | 
cls.predictions.transform.LayerNorm.weight | UNEXPECTED | 
cls.predictions.transform.dense.weight     | UNEXPECTED | 
cls.predictions.transform.dense.bias       | UNEXPECTED | 
cls.predictions.transform.LayerNorm.bias   | UNEXPECTED | 
cls.seq_relationship.bias                  | UNEXPECTED | 
cls.predictions.bias                       | UNEXPECTED | 
classifier.bias                            | MISSING    | 
classifier.weight                          | MISSING    | 

Notes:
- UNEXPECTED:	can be ignored when loading from different task/architecture; not ok if you expect identical arch.
- MISSING:	those params were newly initialized because missing from the checkpoint. Consider training on your downstream task.


Epoch,Training Loss,Validation Loss,Accuracy,F1
1,0.471555,0.398554,0.799020,0.846442
2,0.317997,0.382299,0.845588,0.893039
3,0.169479,0.492709,0.843137,0.882353
4,0.112328,0.609630,0.865196,0.905660
5,0.030664,0.665817,0.867647,0.903571
6,0.033833,0.865115,0.855392,0.898451
7,0.028753,0.954503,0.850490,0.896785
8,0.003791,0.878268,0.855392,0.897747
9,0.000345,0.943451,0.862745,0.904110
10,0.000318,0.931049,0.860294,0.902230


Training Loss,Validation Loss,Epoch,Accuracy,F1
0.000318,0.931049,10,0.860294,0.902230


{'eval_loss': 0.9310493469238281,
 'eval_accuracy': 0.8602941176470589,
 'eval_f1': 0.902229845626072}

## 4. Перевірка на власних прикладах

In [17]:
def predict(model, sentence1, sentence2):
    model.eval()
    inputs = tokenizer(sentence1, sentence2, return_tensors="pt").to(model.device)
    with torch.no_grad():
        probs = model(**inputs).logits.softmax(dim=-1)[0]
    return f"парафраз: {probs[1]:.1%}"


pairs = [
    ("The company reported a profit of $5 million.", "The firm earned five million dollars in profit."),
    ("The company reported a profit of $5 million.", "The weather was sunny all week."),
    ("She bought a new car yesterday.", "Yesterday she purchased a new vehicle."),
]
for s1, s2 in pairs:
    print(f"{s1}\n{s2}\n→ {predict(model, s1, s2)}\n")

The company reported a profit of $5 million.
The firm earned five million dollars in profit.
→ парафраз: 86.0%

The company reported a profit of $5 million.
The weather was sunny all week.
→ парафраз: 2.6%

She bought a new car yesterday.
Yesterday she purchased a new vehicle.
→ парафраз: 99.0%



In [18]:
# Завдання 10: 4 додаткові перевірки на власних даних
my_pairs = [
    ("The engineering team fixed the critical bug in less than an hour.",
     "A severe software issue was resolved by the developers within sixty minutes."),
    ("Artificial intelligence is transforming modern healthcare diagnostics.",
     "My favorite Italian restaurant serves the best margherita pizza in town."),
    ("The startup acquired the tech giant for two billion dollars.",
     "The tech giant acquired the startup for two billion dollars."),
    ("Due to heavy snowfall, all flights from the international airport were canceled.",
     "Severe winter weather forced the airport to suspend all scheduled departures."),
]
for s1, s2 in my_pairs:
    print(f"{s1}\n{s2}\n→ {predict(model, s1, s2)}\n")

The engineering team fixed the critical bug in less than an hour.
A severe software issue was resolved by the developers within sixty minutes.
→ парафраз: 69.7%

Artificial intelligence is transforming modern healthcare diagnostics.
My favorite Italian restaurant serves the best margherita pizza in town.
→ парафраз: 2.9%

The startup acquired the tech giant for two billion dollars.
The tech giant acquired the startup for two billion dollars.
→ парафраз: 99.3%

Due to heavy snowfall, all flights from the international airport were canceled.
Severe winter weather forced the airport to suspend all scheduled departures.
→ парафраз: 62.0%



## Відповіді на теоретичні питання (`01_bert_head.ipynb`)

### 1. Як довго виконувалися клітинки ноутбуку `01_bert_head`? Чи були проблеми з запуском, які?
* **Час виконання:** Загальний час виконання всіх клітинок на GPU T4 склав близько **1.5–2 хвилин**. Безпосередньо цикл донавчання (`trainer.train()` на 3 епохи, 690 кроків) зайняв **1 хвилину 01 секунду**, а решта часу пішла на завантаження датасету MRPC, ваг моделі `bert-base-uncased` (~440 МБ) та токенізацію.
* **Проблеми та попередження:** Критичних помилок при запуску не виникло. З'явилися лише стандартні інформаційні повідомлення:
  1. `Warning: You are sending unauthenticated requests to the HF Hub` — попередження про відсутність токена `HF_TOKEN` (не заважає завантаженню відкритих моделей).
  2. Звіт `BertForSequenceClassification LOAD REPORT` про ігнорування старих ваг голови `cls.*` (`UNEXPECTED`) та випадкову ініціалізацію нових ваг `classifier.weight` і `classifier.bias` (`MISSING`), що є штатною поведінкою при заміні голови моделі.

### 2. Як Ви розумієте, що таке токен `[CLS]`? Яким чином він виконує свою функцію?
* **Що це:** `[CLS]` (від *Classification*) — це спеціальний службовий токен, який токенізатор BERT автоматично ставить на **першу позицію** (індекс 0) кожної вхідної послідовності або пари речень (`[CLS] речення 1 [SEP] речення 2 [SEP]`).
* **Як виконує функцію:** На відміну від звичайних слів, він не має власного лексичного значення. Проходячи крізь 12 шарів двонапрямленого механізму уваги (*bidirectional self-attention*), токен `[CLS]` «звертає увагу» на всі інші токени обох речень і акумулює в своєму прихованому стані контекстне представлення всієї вхідної послідовності. На виході останнього шару вектор цього токена (розмірністю 768) пропускається через шар `pooler` та подається на лінійну голову-класифікатор для формування прогнозу.

### 3. Яке пряме застосування базової моделі `bert-base-uncased`?
* **Пряме застосування «з коробки»:** Базова модель попередньо натренована на двох задачах самокерованого навчання (*self-supervised learning*):
  1. **Masked Language Modeling (MLM / `fill-mask`)** — передбачення замаскованих токенів `[MASK]` у реченні на основі лівого і правого контексту.
  2. **Next Sentence Prediction (NSP)** — визначення того, чи є друге речення логічним продовженням першого в оригінальному тексті.
* **Основне практичне призначення:** Використання як універсального **екстрактора контекстних ознак (backbone)** для подальшого донавчання (*fine-tuning*) під прикладні задачі (класифікація текстів, визначення парафраз, аналіз тональності, NER, відповіді на запитання).

### 4. Як Ви вважаєте, чи варто боротися (і якщо так, то яким чином) з проявами сексизму LLM-моделей?
* **Чи варто боротися:** Однозначно **так**. У демо видно, що для `"The man worked as a [MASK]."` модель пропонує професії *carpenter, waiter, barber, mechanic, salesman*, а для `"The woman worked as a [MASK]."` — *nurse, waitress, maid, prostitute, cook*. Мовна модель віддзеркалює статистичні перекоси та стереотипи сирих інтернет-корпусів. Якщо таку модель вбудувати в HR-систему скринінгу резюме, скорингу або рекомендацій, вона автоматизує та масштабує дискримінацію.
* **Яким чином боротися:**
  * **Курація та балансування даних (Data Curation & Counterfactual Data Augmentation):** очищення тренувальних корпусів та додавання симетричних прикладів із заміною гендерних маркерів (*he/she*, *man/woman*) у контексті різних професій.
  * **Пост-тренувальне вирівнювання (Alignment — SFT, RLHF, DPO):** донавчання моделей на етичних інструкціях і людських преференціях, щоб модель уникала токсичних чи стереотипних узагальнень.
  * **Дебіасинг ембедингів:** математичне проєктування векторних представлень професій ортогонально до гендерної осі в латентному просторі.

### 5. Для чого в секції «Токенізація пари речень» виводяться числа `0` і `1`?
* Ці числа є елементами масиву `token_type_ids` (їх також називають *segment IDs*). Оскільки обидва речення склеюються в один спільний масив `input_ids`, моделі потрібен явний сигнал, де закінчується перше речення і починається друге:
  * `0` позначає токени, що належать до **першого речення** (включно з початковим `[CLS]` та першим розділювачем `[SEP]`).
  * `1` позначає токени, що належать до **другого речення** (включно з фінальним `[SEP]`).

### 6. Що позначають числа `768` і `2`, що виводяться після завантаження `AutoModelForSequenceClassification.from_pretrained()`?
* Вивід `Linear(in_features=768, out_features=2, bias=True)` описує архітектуру нової лінійної голови класифікатора (`model.classifier`):
  * **`768` (`in_features`)** — розмірність вхідного вектора прихованого стану (`hidden_size`) базової моделі `bert-base-uncased` (представлення токена `[CLS]`, яке надходить із кістяка BERT).
  * **`2` (`out_features`)** — кількість вихідних нейронів (логітів), що відповідає кількості класів у нашій бінарній задачі (`num_labels=2`: `0` — *not_equivalent*, `1` — *equivalent*).

### 7. Те що в кістяку (backbone) набагато більше параметрів ніж в голові, це добре чи погано в нашому випадку?
* У кістяку `109,482,240` параметрів, а в голові лише `1,538` ($768 \times 2 + 2$) — і для задачі переносу навчання (*transfer learning*) на невеликому датасеті MRPC (3668 тренувальних пар) **це дуже добре**:
  1. **Потужна репрезентація в кістяку:** 109.5 млн попередньо навчених параметрів уже вміють будувати глибокі семантичні та синтаксичні представлення англійської мови.
  2. **Простота та стійкість голови:** Оскільки нова голова ініціалізується випадковими числами, мала кількість параметрів у ній (1538) дозволяє швидко налаштувати межу прийняття рішень поверх готових ознак BERT без миттєвого перенавчання.

### 8. Що саме вимірюється за допомогою метрик `accuracy` & `F1` в `01_bert_head`?
* **`Accuracy` (Частка правильних відповідей):** вимірює відношення кількості правильно класифікованих пар речень (і парафраз, і не-парафраз) до загальної кількості пар у валідаційній вибірці (408 прикладів):
  $$\text{Accuracy} = \frac{TP + TN}{TP + TN + FP + FN}$$
* **`F1-score`:** гармонійне середнє між точністю (*Precision* — частка реальних парафраз серед усіх пар, які модель назвала парафразами) та повнотою (*Recall* — частка знайдених парафраз серед усіх реальних парафраз у датасеті) для позитивного класу (`label = 1`):
  $$F_1 = 2 \cdot \frac{\text{Precision} \cdot \text{Recall}}{\text{Precision} + \text{Recall}}$$
* **Чому потрібні обидві:** Датасет MRPC незбалансований (~68% прикладів мають мітку `1` — *equivalent*). Якби модель завжди передбачала `1`, її `accuracy` вже складала б ~68%, тому `F1` дає значно об'єктивнішу оцінку якості розпізнавання цільового класу.

### 9. Збільшіть кількість епох тренування в `01_bert_head`, покажіть (скопіювавши результати) що відбувається і чому.
* Результати при збільшенні `num_train_epochs` з `3` до `10` (`[2300/2300 03:23, Epoch 10/10]`):

| Epoch | Training Loss | Validation Loss | Accuracy | F1 |
| :--- | :--- | :--- | :--- | :--- |
| 1 | 0.471555 | 0.398554 | 0.799020 | 0.846442 |
| 2 | 0.317997 | 0.382299 | 0.845588 | 0.893039 |
| 3 | 0.169479 | 0.492709 | 0.843137 | 0.882353 |
| 4 | 0.112328 | 0.609630 | 0.865196 | 0.905660 |
| 5 | 0.030664 | 0.665817 | 0.867647 | 0.903571 |
| 6 | 0.033833 | 0.865115 | 0.855392 | 0.898451 |
| 7 | 0.028753 | 0.954503 | 0.850490 | 0.896785 |
| 8 | 0.003791 | 0.878268 | 0.855392 | 0.897747 |
| 9 | 0.000345 | 0.943451 | 0.862745 | 0.904110 |
| 10 | 0.000318 | 0.931049 | 0.860294 | 0.902230 |

* Фінальний результат `trainer_10.evaluate()`:
  `{'eval_loss': 0.9310493469238281, 'eval_accuracy': 0.8602941176470589, 'eval_f1': 0.902229845626072}`

* **Що відбувається і чому:**
  * Спостерігається класичне **перенавчання (overfitting)**.
  * `Training Loss` падає майже до нуля (з `0.471555` на 1-й епосі до `0.000318` на 10-й епосі), тоді як `Validation Loss` досягає мінімуму на 2-й епосі (`0.382299`), після чого стрімко зростає до `0.931049`–`0.954503`. Метрики `Accuracy` та `F1` досягають максимуму на 4–5 епохах (`86.76%` та `90.57%`), після чого перестають покращуватися і коливаються на плато (`86.03%` та `90.22%` на 10-й епосі).
  * **Причина:** Модель на 109.5 млн параметрів має надлишкову ємність для невеликого датасету з 3668 тренувальних прикладів. Після 2–4 епох модель просто «зазубрює» тренувальну вибірку і стає надмірно впевненою у своїх прогнозах (видає екстремальні логіти), через що функція втрат крос-ентропії дуже сильно штрафує її за кожну помилку на валідаційних даних.

### 10. Додайте ще 4 перевірки на власних даних в пункті 4 в `01_bert_head`.
* **Коментар до результатів:** Модель правильно розпізнала перефразовані речення з синонімами (`69.7%` та `62.0%` > `50%`) і чітко відсіяла речення на різні теми (`2.9%`). Водночас третій приклад (`99.3%`) наочно демонструє відому вразливість класифікаторів парафраз на базі BERT (Shortcut Learning / Bag-of-Words bias): коли два речення складаються з абсолютно однакових слів, але мають протилежний зміст через зміну порядку підмета і додатка (*хто кого купив*), модель орієнтується на 100% лексичний збіг і помилково вважає їх стовідсотковим парафразом.